# Unit 7 — SoccerTwos (MA-POCA / Unity ML-Agents)

Notebook corregido para el hands-on de Unit 7 del Hugging Face Deep RL Course.

**Objetivo:** entrenar un agente para `SoccerTwos`, generar un archivo `.onnx` y subirlo al Hub en:

`Lizeth-otalora07/poca-SoccerTwos`

> Nota: el curso indica que para validar esta práctica basta con subir un modelo entrenado; no exige un resultado mínimo. El entrenamiento completo recomendado puede tardar varias horas, por eso aquí se deja una configuración corta para Colab.


## 1. Crear entorno Python 3.10.12

ML-Agents no funciona bien con Python 3.12/3.13 en Colab. Por eso creamos un Python 3.10.12 separado en `/content/py310`.


In [1]:
!rm -rf /content/miniconda /content/py310 miniconda.sh

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -f -p /content/miniconda

!/content/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/content/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

!/content/miniconda/bin/conda create -y -p /content/py310 python=3.10.12 pip

PREFIX=/content/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/miniconda
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | / - \ | / - \ | done
Channels:
 - defaults
Platform: linux-64
Solving environment: \ done


==> WARNING: A newer version of conda exists. <==
    current version: 26.3.2
    latest version: 26.5.0

Please update co

In [2]:
!/content/py310/bin/python --version

Python 3.10.12


## 2. Instalar herramientas del sistema

In [3]:
!apt-get update -qq
!apt-get install -y git wget unzip git-lfs

!git lfs install

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
unzip is already the newest version (6.0-26ubuntu3.2).
wget is already the newest version (1.21.2-2ubuntu1.1).
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.
Git LFS initialized.


## 3. Clonar e instalar ML-Agents

Esta celda borra y vuelve a clonar `/content/ml-agents`. **No la vuelvas a correr después de descargar SoccerTwos o después de entrenar**, porque borraría archivos generados.


In [4]:
%cd /content

!rm -rf ml-agents
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents

%cd /content/ml-agents

!/content/py310/bin/python -m pip install --upgrade pip setuptools wheel
!/content/py310/bin/python -m pip install -e ./ml-agents-envs
!/content/py310/bin/python -m pip install -e ./ml-agents
!/content/py310/bin/python -m pip install huggingface_hub gdown

/content
Cloning into 'ml-agents'...
remote: Enumerating objects: 2442, done.
remote: Counting objects: 100% (2442/2442), done.
remote: Compressing objects: 100% (1814/1814), done.
remote: Total 2442 (delta 904), reused 1652 (delta 608), pack-reused 0 (from 0)
Receiving objects: 100% (2442/2442), 97.55 MiB | 29.90 MiB/s, done.
Resolving deltas: 100% (904/904), done.
Updating files: 100% (2362/2362), done.
/content/ml-agents
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.7 MB/s  0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.46.3
    Uninstalling wheel-0.46.3:
      Successfully uninstalled wheel-0.46.3
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pip]
Obtaining file:///content/ml-agents/ml-agents-envs
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... do

In [5]:
!/content/py310/bin/mlagents-learn --help

usage: mlagents-learn
       [-h]
       [--env ENV_PATH]
       [--resume]
       [--deterministic]
       [--force]
       [--run-id RUN_ID]
       [--initialize-from RUN_ID]
       [--seed SEED]
       [--inference]
       [--base-port BASE_PORT]
       [--num-envs NUM_ENVS]
       [--num-areas NUM_AREAS]
       [--debug]
       [--env-args ...]
       [--max-lifetime-restarts MAX_LIFETIME_RESTARTS]
       [--restarts-rate-limit-n RESTARTS_RATE_LIMIT_N]
       [--restarts-rate-limit-period-s RESTARTS_RATE_LIMIT_PERIOD_S]
       [--torch]
       [--tensorflow]
       [--results-dir RESULTS_DIR]
       [--timeout-wait TIMEOUT_WAIT]
       [--width WIDTH]
       [--height HEIGHT]
       [--quality-level QUALITY_LEVEL]
       [--time-scale TIME_SCALE]
       [--target-frame-rate TARGET_FRAME_RATE]
       [--capture-frame-rate CAPTURE_FRAME_RATE]
       [--no-graphics]
       [--no-graphics-monitor]
       [--torch-device DEVICE]
       [trainer_config_path]

positional arguments:
  trai

## 4. Descargar SoccerTwos correctamente

La descarga anterior fallaba porque bajaba un HTML pequeño en vez del ZIP real. Esta versión usa `gdown` desde Python y valida que el archivo sea un ZIP real.


In [8]:
%cd /content/ml-agents

!rm -rf training-envs-executables SoccerTwos.zip
!mkdir -p training-envs-executables

# Instalar gdown en el Python 3.10 que estamos usando
!/content/py310/bin/python -m pip install -q --upgrade gdown

# Descargar SoccerTwos desde Google Drive usando el ID directamente
!/content/py310/bin/python -m gdown "1KuqBKYiXiIcU4kNMqEzhgypuFP5_45CL" -O SoccerTwos.zip

# Verificar que sí se descargó algo real
!ls -lh SoccerTwos.zip
!file SoccerTwos.zip

# Validar que sea un ZIP real y no una página HTML
!/content/py310/bin/python -c "import os, zipfile; path='SoccerTwos.zip'; size=os.path.getsize(path); print('Tamaño ZIP:', size, 'bytes'); assert size > 1000000, 'El archivo es demasiado pequeño: no es el ZIP real'; assert zipfile.is_zipfile(path), 'No es un ZIP válido'; print('ZIP válido.')"

# Descomprimir
!unzip -o SoccerTwos.zip -d training-envs-executables/
!chmod -R 755 training-envs-executables

/content/ml-agents
Downloading...
From (original): https://drive.google.com/uc?id=1KuqBKYiXiIcU4kNMqEzhgypuFP5_45CL
From (redirected): https://drive.google.com/uc?id=1KuqBKYiXiIcU4kNMqEzhgypuFP5_45CL&confirm=t&uuid=7e013668-f9cc-4a1d-8830-64a260c9b7fa
To: /content/ml-agents/SoccerTwos.zip
100% 37.0M/37.0M [00:01<00:00, 34.5MB/s]
-rw------- 1 root root 36M Feb  2  2023 SoccerTwos.zip
SoccerTwos.zip: Zip archive data, at least v1.0 to extract, compression method=store
Tamaño ZIP: 36963480 bytes
ZIP válido.
Archive:  SoccerTwos.zip
   creating: training-envs-executables/SoccerTwos_Data/
  inflating: training-envs-executables/SoccerTwos_Data/app.info  
  inflating: training-envs-executables/SoccerTwos_Data/boot.config  
  inflating: training-envs-executables/SoccerTwos_Data/globalgamemanagers  
  inflating: training-envs-executables/SoccerTwos_Data/globalgamemanagers.assets  
  inflating: training-envs-executables/SoccerTwos_Data/level0  
  inflating: training-envs-executables/SoccerTwos_D

## 5. Verificar ejecutable de SoccerTwos

In [10]:
%cd /content/ml-agents

!find training-envs-executables -maxdepth 5 -type f | head -50
!find training-envs-executables -type f -perm /111

/content/ml-agents
training-envs-executables/SoccerTwos_Data/sharedassets0.assets
training-envs-executables/SoccerTwos_Data/globalgamemanagers.assets
training-envs-executables/SoccerTwos_Data/Plugins/lib_burst_generated.so
training-envs-executables/SoccerTwos_Data/Plugins/libgrpc_csharp_ext.x64.so
training-envs-executables/SoccerTwos_Data/RuntimeInitializeOnLoads.json
training-envs-executables/SoccerTwos_Data/globalgamemanagers
training-envs-executables/SoccerTwos_Data/sharedassets0.assets.resS
training-envs-executables/SoccerTwos_Data/resources.assets.resS
training-envs-executables/SoccerTwos_Data/ScriptingAssemblies.json
training-envs-executables/SoccerTwos_Data/level0.resS
training-envs-executables/SoccerTwos_Data/resources.assets
training-envs-executables/SoccerTwos_Data/Managed/Unity.TextMeshPro.dll
training-envs-executables/SoccerTwos_Data/Managed/Unity.ML-Agents.CommunicatorObjects.dll
training-envs-executables/SoccerTwos_Data/Managed/UnityEngine.ClothModule.dll
training-envs-ex

In [11]:
%cd /content/ml-agents

import os

candidates = []
for root, dirs, files in os.walk("training-envs-executables"):
    for name in files:
        path = os.path.join(root, name)
        if os.access(path, os.X_OK) and (
            "SoccerTwos" in name or name.endswith(".x86_64") or name.endswith(".exe")
        ):
            candidates.append(path)

print("Candidatos:")
for c in candidates:
    print("-", c)

if not candidates:
    raise FileNotFoundError("No se encontró el ejecutable de SoccerTwos. Revisa la descarga y la descompresión.")

SOCCER_EXECUTABLE = candidates[0]
print("Ejecutable seleccionado:", SOCCER_EXECUTABLE)

/content/ml-agents
Candidatos:
- training-envs-executables/SoccerTwos.x86_64
Ejecutable seleccionado: training-envs-executables/SoccerTwos.x86_64


## 6. Crear configuración MA-POCA

El curso usa `trainer_type: poca` con `self_play`. Para que Colab no tarde 5–8 horas, aquí usamos `max_steps: 200000`.

Si quieres entrenamiento completo, cambia `max_steps` a `5000000`, pero tardará varias horas.


In [12]:
%cd /content/ml-agents
!mkdir -p config/poca

/content/ml-agents


In [13]:
%%writefile /content/ml-agents/config/poca/SoccerTwos.yaml
behaviors:
  SoccerTwos:
    trainer_type: poca
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: constant
    network_settings:
      normalize: false
      hidden_units: 512
      num_layers: 2
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.99
        strength: 1.0
    keep_checkpoints: 5
    checkpoint_interval: 50000
    max_steps: 200000
    time_horizon: 1000
    summary_freq: 10000
    self_play:
      save_steps: 50000
      team_change: 200000
      swap_steps: 2000
      window: 10
      play_against_latest_model_ratio: 0.5
      initial_elo: 1200.0

Overwriting /content/ml-agents/config/poca/SoccerTwos.yaml


In [14]:
!cat /content/ml-agents/config/poca/SoccerTwos.yaml

behaviors:
  SoccerTwos:
    trainer_type: poca
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: constant
    network_settings:
      normalize: false
      hidden_units: 512
      num_layers: 2
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.99
        strength: 1.0
    keep_checkpoints: 5
    checkpoint_interval: 50000
    max_steps: 200000
    time_horizon: 1000
    summary_freq: 10000
    self_play:
      save_steps: 50000
      team_change: 200000
      swap_steps: 2000
      window: 10
      play_against_latest_model_ratio: 0.5
      initial_elo: 1200.0


## 7. Entrenar SoccerTwos

Cuando empiece el entrenamiento, deberías ver mensajes como `Connected to Unity environment`, `SoccerTwos` y luego pasos de entrenamiento.

No cierres Colab mientras esté entrenando.


In [15]:
%cd /content/ml-agents

!/content/py310/bin/mlagents-learn ./config/poca/SoccerTwos.yaml \
  --env="{SOCCER_EXECUTABLE}" \
  --run-id="SoccerTwos" \
  --no-graphics \
  --force

/content/ml-agents

            ┐  ╖
        ╓╖╬│╡  ││╬╖╖
    ╓╖╬│││││┘  ╬│││││╬╖
 ╖╬│││││╬╜        ╙╬│││││╖╖                               ╗╗╗
 ╬╬╬╬╖││╦╖        ╖╬││╗╣╣╣╬      ╟╣╣╬    ╟╣╣╣             ╜╜╜  ╟╣╣
 ╬╬╬╬╬╬╬╬╖│╬╖╖╓╬╪│╓╣╣╣╣╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╒╣╣╖╗╣╣╣╗   ╣╣╣ ╣╣╣╣╣╣ ╟╣╣╖   ╣╣╣
 ╬╬╬╬┐  ╙╬╬╬╬│╓╣╣╣╝╜  ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╣╙ ╙╣╣╣  ╣╣╣ ╙╟╣╣╜╙  ╫╣╣  ╟╣╣
 ╬╬╬╬┐     ╙╬╬╣╣      ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣     ╣╣╣┌╣╣╜
 ╬╬╬╜       ╬╬╣╣      ╙╝╣╣╬      ╙╣╣╣╗╖╓╗╣╣╣╜ ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣╦╓    ╣╣╣╣╣
 ╙   ╓╦╖    ╬╬╣╣   ╓╗╗╖            ╙╝╣╣╣╣╝╜   ╘╝╝╜   ╝╝╝  ╝╝╝   ╙╣╣╣    ╟╣╣╣
   ╩╬╬╬╬╬╬╦╦╬╬╣╣╗╣╣╣╣╣╣╣╝                                             ╫╣╣╣╣
      ╙╬╬╬╬╬╬╬╣╣╣╣╣╣╝╜
          ╙╬╬╬╣╣╣╜
             ╙
        
 Version information:
  ml-agents: 1.2.0.dev0,
  ml-agents-envs: 1.2.0.dev0,
  Communicator API: 1.5.0,
  PyTorch: 2.8.0+cu128
[INFO] Connected to Unity environment with package version 2.3.0-exp.3 and communication version 1.5.0
[INFO] Connected new bra

## 8. Verificar modelo generado

In [16]:
%cd /content/ml-agents

!find results/SoccerTwos -name "*.onnx" -o -name "*.pt" | head -20

/content/ml-agents
results/SoccerTwos/SoccerTwos.onnx
results/SoccerTwos/SoccerTwos/SoccerTwos-149938.onnx
results/SoccerTwos/SoccerTwos/SoccerTwos-199350.onnx
results/SoccerTwos/SoccerTwos/SoccerTwos-49603.pt
results/SoccerTwos/SoccerTwos/SoccerTwos-49603.onnx
results/SoccerTwos/SoccerTwos/SoccerTwos-99834.onnx
results/SoccerTwos/SoccerTwos/SoccerTwos-199350.pt
results/SoccerTwos/SoccerTwos/SoccerTwos-99834.pt
results/SoccerTwos/SoccerTwos/SoccerTwos-200350.pt
results/SoccerTwos/SoccerTwos/SoccerTwos-149938.pt
results/SoccerTwos/SoccerTwos/SoccerTwos-200350.onnx
results/SoccerTwos/SoccerTwos/checkpoint.pt


## 9. Login en Hugging Face

In [17]:
from huggingface_hub import notebook_login

notebook_login()

!git config --global credential.helper store

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 10. Subir SoccerTwos a Hugging Face Hub

Repositorio esperado:

`Lizeth-otalora07/poca-SoccerTwos`


In [18]:
%cd /content/ml-agents

!/content/py310/bin/mlagents-push-to-hf \
  --run-id="SoccerTwos" \
  --local-dir="./results/SoccerTwos" \
  --repo-id="Lizeth-otalora07/poca-SoccerTwos" \
  --commit-message="Upload POCA SoccerTwos trained agent"

/content/ml-agents
[INFO] This function will create a model card and upload your SoccerTwos into HuggingFace Hub. This is a work in progress: If you encounter a bug, please send open an issue
[INFO] Pushing repo SoccerTwos to the Hugging Face Hub
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...38068.82b2b6f7f4bc.5945.0:   1% 2.12k/148k [00:00<?, ?B/s]


  ...wos/SoccerTwos-49603.onnx:   1% 25.3k/1.77M [00:00<?, ?B/s]



  ...os/SoccerTwos-200350.onnx:   1% 25.3k/1.77M [00:00<?, ?B/s]




  ...rTwos/SoccerTwos-49603.pt:   1% 407k/28.4M [00:00<?, ?B/s]





  ...wos/SoccerTwos-99834.onnx:   1% 25.3k/1.77M [00:00<?, ?B/s]






  ...occerTwos/SoccerTwos.onnx:   1% 25.3k/1.77M [00:00<?, ?B/s]







  ...rTwos/SoccerTwos-99834.pt:   0% 21.8k/28.4M [00:00<?, ?B/s]








  ...os/SoccerTwos-149938.onnx:   1% 25.3k/1.77M [00:00<?, ?B/s]









  ...Twos/SoccerTwos-199350.pt:   1% 282k/28

## 11. Verificación final del repo

El curso pide revisar que el repo tenga:

1. El tag `ML-Agents-SoccerTwos`.
2. Un archivo `SoccerTwos.onnx`.

Esta celda lista los archivos del repo después del push.


In [19]:
from huggingface_hub import list_repo_files

repo_id = "Lizeth-otalora07/poca-SoccerTwos"
files = list_repo_files(repo_id)

print("Archivos en el repo:")
for f in files:
    print("-", f)

print("\nArchivos ONNX:")
for f in files:
    if f.endswith(".onnx"):
        print("-", f)

Archivos en el repo:
- .gitattributes
- README.md
- SoccerTwos.onnx
- SoccerTwos/SoccerTwos-149938.onnx
- SoccerTwos/SoccerTwos-149938.pt
- SoccerTwos/SoccerTwos-199350.onnx
- SoccerTwos/SoccerTwos-199350.pt
- SoccerTwos/SoccerTwos-200350.onnx
- SoccerTwos/SoccerTwos-200350.pt
- SoccerTwos/SoccerTwos-49603.onnx
- SoccerTwos/SoccerTwos-49603.pt
- SoccerTwos/SoccerTwos-99834.onnx
- SoccerTwos/SoccerTwos-99834.pt
- SoccerTwos/checkpoint.pt
- SoccerTwos/events.out.tfevents.1779238068.82b2b6f7f4bc.5945.0
- config.json
- configuration.yaml
- run_logs/Player-0.log
- run_logs/timers.json
- run_logs/training_status.json

Archivos ONNX:
- SoccerTwos.onnx
- SoccerTwos/SoccerTwos-149938.onnx
- SoccerTwos/SoccerTwos-199350.onnx
- SoccerTwos/SoccerTwos-200350.onnx
- SoccerTwos/SoccerTwos-49603.onnx
- SoccerTwos/SoccerTwos-99834.onnx
